This notebook builds an Agentic system to summarize earnings calls using simple chaining. I could have using routing as well, but the logic is simple enough to not need it.
In the process, I also wanted to (a) check how effective it is to use LLM as a critic in a loop over just multi-shot prompting, and
(b)generalize the system to it can work for any example of iteratively improving a draft using a critic and multi-shot examples
Therefore the set up is as follows:
(a) We use a few Agent personas: Extractor (turns 'target' artifacts into a rubric), Generator (creates a first draft using source material), Critic (grades drafts and offers suggestions to improve using
the rubric and the source material), Improver (improves on the draft using the critic's suggestions)
(b) We set up a Domain configurable by the systeme prompts we define for these persona
(c) The Agentic System has 3 classes - a State (with the context, source material, rubric and succcessive drafts), an Agents (to call the llm with prompts and return output), and Orchestration (to manage the workflow - target artifacts -> rubric, source material -> draft -> critic eval/feedback -> improve).
The implementation below also compares the Agentic approach with a simple multi-shot and the difference is clear.

In [ ]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

In [ ]:
google_api_key = os.getenv('GOOGLE_API_KEY')
gemini = OpenAI(api_key=google_api_key, base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
model_name = "gemini-2.5-flash"

In [ ]:
import json
import re 

class FrameworkState:
    # Maintains the state of the Agentic system
    def __init__(self, domain_context):
        self.context = domain_context   # the domain e.g., creating class notes from a transcript
        self.source_material = None     # the raw input e.g., class lecture transcript
        self.rubric = None              # the criteria to evaluate the output
        self.history = []               # successive output drafts with iteration num, critic feedback & score

    def log_iteration(self, loop, artifact, score, feedback):
        # Update the history with latest iteration
        self.history.append({
            "loop": loop, "artifact": artifact, 
            "score": score, "feedback": feedback
        })

    def get_best_artifact(self):
        # Sort history by score and return the highest scoring artifact
        return max(self.history, key=lambda x: x["score"])["artifact"] if self.history else None


class AgentRegistry:
    # Access domain context for prompts
    def __init__(self, state: FrameworkState):
        self.ctx = state.context  

    def _call_llm(self, system_prompt, user_prompt):
        response = gemini.chat.completions.create(
            model=model_name,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt }
            ]
        )
        print(f"  -> [LLM Called as: {system_prompt.split('.')[0]}]")
        return response

    def extractor(self, examples):
        # Extract a good evaluation rubric from the examples
        sys = f"You are a {self.ctx['extractor_persona']}. Extract a rubric."
        res = self._call_llm(sys, f"Examples: {examples}")
        # Extract text content from ChatCompletion object
        return res.choices[0].message.content

    def generator(self, source, rubric):
        # Generate a first draft from source material keeping the rubric in mind 
        sys = f"You are a {self.ctx['generator_persona']}. Generate {self.ctx['target_artifact']}."
        res = self._call_llm(sys, f"Source: {source}\nRubric: {rubric}")
        # Extract text content from ChatCompletion object
        return res.choices[0].message.content

    def critic(self, source, artifact, rubric):
        # Evaluate a draft providing a score and feedback
        sys = f"You are a {self.ctx['critic_persona']}. Evaluate the {self.ctx['target_artifact']}."
        usr = f"Source: {source}\nArtifact: {artifact}\nRubric: {rubric}. Output 'score' and 'feedback' in strict json format"
        res = self._call_llm(sys, usr)
        
        # Pull text from the object
        raw_text = res.choices[0].message.content
        
        # Robust parsing to handle chat formatting
        try:
            # Find the true JSON boundaries
            match = re.search(r'\{.*\}', raw_text, re.DOTALL)
            if match:
                data = json.loads(match.group(0))
                return int(data["score"]), data["feedback"]
            else:
                raise ValueError("No JSON payload structure detected in the response.")
        
        except Exception as e:
            print(f"  -> [Warning: JSON parsing failed. Executing safety fallback.]")
            # Fallback to prevent orchestration loop failure
            return 0, f"Failed to parse model response natively. Raw text: {raw_text}"

    def improver(self, source, artifact, feedback):
        # Improve a draft using both the source and the feedback
        sys = f"You are an {self.ctx['improver_persona']}. Fix the {self.ctx['target_artifact']}."
        res = self._call_llm(sys, f"Draft: {artifact}\nFeedback: {feedback}")
        # FIXED: text content from ChatCompletion object
        return res.choices[0].message.content


class RefinementOrchestrator:
    # Orchestrates the entire System
    def __init__(self, state: FrameworkState, agents: AgentRegistry):
        self.state = state
        self.agents = agents

    def compile_rubric(self, examples):
        # Build an evaluation rubric from examples
        print("\n--- Phase 1: Calibrating Rubric ---")
        self.state.rubric = self.agents.extractor(examples)

    def process(self, source_material, max_loops=3, threshold=90):
        # Develop the first draft, then refine
        print("\n--- Phase 2: Generating Initial Draft ---")
        self.state.source_material = source_material
        artifact = self.agents.generator(source_material, self.state.rubric)

        print("\n--- Phase 3: Entering Reflection Loop ---")
        for loop in range(max_loops):
            print(f"Loop {loop + 1}: Critiquing...")
            score, feedback = self.agents.critic(source_material, artifact, self.state.rubric)
            self.state.log_iteration(loop, artifact, score, feedback)
            print(f"Score: {score} | Feedback: {feedback}")

            if score >= threshold:
                print(">> Success! Threshold met.")
                return artifact
                
            print(">> Improving Draft...")
            artifact = self.agents.improver(source_material, artifact, feedback)

        print(">> Max loops hit. Returning best draft.")
        return self.state.get_best_artifact()

In [ ]:
# Raw Earnings Call Transcript
transcript = """
Operator: Welcome to the RetailGiant Q4 2025 Earnings Conference Call. 
CEO: Good morning, everyone. Q4 was a milestone quarter for RetailGiant. Total revenue for the quarter reached $24.5 billion, 
marking an impressive 12% growth year-over-year, which sailed past our original internal estimate of $23.8 billion. 
Our digital e-commerce channel was the star of the show, pulling in $9.8 billion of that total, which represents a massive 30% surge compared to Q4 last year. 
Physical store sales were flat at $14.7 billion, showing a minor 0.5% drop as foot traffic slightly normalized post-holiday. 
Our gross margin landed at 38.5%. While healthy, this is down 150 basis points from last year. 
This compression was explicitly driven by aggressive promotional discounting required to clear out 
an oversupply of winter apparel, alongside rising domestic labor costs in our fulfillment centers. 
Diluted EPS for the quarter was $2.40, beating Wall Street consensus of $2.25. 
CFO: Looking ahead into fiscal year 2026, we are maintaining a cautious but optimistic stance. 
We expect significant macro headwinds from persistent ocean freight inflation and a cooling retail consumer environment, 
which we anticipate will continue to pressure our brick-and-mortar margins by an additional 40 to 60 basis points in the first half of the year. 
For full-year FY2026 guidance, we are projecting total revenue to land in the range of $96 billion to $98 billion. 
We expect capital expenditures to top $3.2 billion as we invest heavily in expanding our automated fulfillment nodes. 
We are now open for questions.
"""

# Few-Shot Examples
example1 = """
# Executive Earnings Briefing: TechCorp Q3 2026
## 1. Core Financial Metrics
* Revenue: $12.4B (+8% YoY) vs. $12.1B expected.
* Gross Margin: 64.2% (contracted 80 bps YoY due to supply chain inflation).
* Diluted EPS: $1.12 vs. $1.05 consensus estimate.
## 2. Segment Performance
* Cloud Infrastructure: $4.8B revenue (+22% YoY). Driven by accelerated enterprise migration.
* Consumer Hardware: $7.6B revenue (-1% YoY). Impacted by cyclical upgrade delays.
## 3. Macro Headwinds & Forward Guidance
* Supply Chain: Freight and logistics costs expected to pressure hardware margins by 50-100 bps into Q4.
* FY2026 Guidance: Revenue projected between $49B - $51B; mapping to a modest 6% full-year growth outlook.
"""

example2 = """
# Executive Earnings Briefing: BioHealth Corp Q2 2026
## 1. Core Financial Metrics
* Revenue: $5.2B (+15% YoY) matches internal projections.
* Gross Margin: 71.0% (+120 bps YoY optimization via manufacturing automation).
* Diluted EPS: $0.85 vs. $0.82 expected.
## 2. Segment Performance
* Therapeutics Pipeline: $3.9B revenue (+18% YoY), led by strong oncology adoption.
* Diagnostic Systems: $1.3B revenue (+4% YoY), showing stable institutional demand.
## 3. Macro Headwinds & Forward Guidance
* Regulatory Delays: Ongoing FDA review extension for 'Compound X' pushes expected commercialization from Q4 2026 to Q2 2027.
* FY2026 Guidance: Reaffirming full-year revenue guidance of $21.5B - $22.0B.
"""

In [ ]:
# simple few-shot prompting for comparison
sys = f"""
You are aHead of Investor Relations expert at distilling earnings call transcripts into concise Executive briefings.
Here are good examples of briefings for you to follow: 
{example1}
{example2}
"""
response = gemini.chat.completions.create(
    model=model_name,
    messages = [
        {"role": "system", "content": sys},
        {"role": "user", "content": f"Create an Executive briefing from this transcript \n {transcript}"}
    ]
)
briefing = response.choices[0].message.content
display(Markdown(briefing))

In [ ]:
# Define the Domain Context
domain_context = {
    "target_artifact": "Executive Earnings Briefing (including Core Metrics, Segment Performance, and Headwinds/Guidance)",
    "extractor_persona": "Head of Investor Relations expert at distilling multi-page corporate performance histories into rigid, quantitative evaluation rubrics.",
    "generator_persona": "Equity Research Analyst skilled at extracting primary financial data and structural talking points from raw transcripts.",
    "critic_persona": "Skeptical Hedge Fund Portfolio Manager who aggressively checks summaries against raw source text to find missing risk factors, guidance adjustments, or numerical omissions.",
    "improver_persona": "Senior Financial Editor specialized in reconciling draft summaries with precise critical feedback and raw transcript data to maximize data density."
}

# Instantiate the Agent System
state = FrameworkState(domain_context)
agents = AgentRegistry(state)
orchestrator = RefinementOrchestrator(state, agents)

# Execution look
good_examples = [example1, example2]
orchestrator.compile_rubric(examples=good_examples)
final_notes = orchestrator.process(source_material=transcript, max_loops=3, threshold=95)

print("\n=== FINAL APPROVED ARTIFACT ===")
display(Markdown(final_notes))